<a href="https://colab.research.google.com/github/srihari-g-acharya-ci109/ML2LAB/blob/main/ML7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import accuracy_score, classification_report

# load dataset
cancer = load_breast_cancer()
X_raw = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target).map({0: "malignant", 1: "benign"})

# pick 5 intuitive features and bin into 3 categories (Low, Mid, High)
selected_feats = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
X = X_raw[selected_feats].copy()

discretizer = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
X_binned = pd.DataFrame(
    discretizer.fit_transform(X).astype(int).astype(str),
    columns=[c.replace(" ", "_") for c in selected_feats]
)
X_binned["target"] = y

print("Discretized feature sample:")
display(X_binned.head())
print("
Class distribution:
", X_binned["target"].value_counts())

In [ ]:
# split train & test
feat_cols = [c for c in X_binned.columns if c != "target"]
train_df, test_df = train_test_split(X_binned, test_size=0.25, random_state=42, stratify=X_binned["target"])

# attempt RIPPER using wittgenstein if installed, otherwise note fallback
try:
    import wittgenstein as lw
    ripper = lw.RIPPER(random_state=42)
    ripper.fit(train_df, class_feat="target", pos_class="malignant")
    print("RIPPER Ruleset:")
    print(ripper.ruleset_)

    X_test_rip = test_df.drop("target", axis=1)
    y_test_rip = test_df["target"]
    rip_preds = ripper.predict(X_test_rip)
    rip_preds = ["malignant" if p in (1, "1", "malignant") else "benign" for p in np.asarray(rip_preds).reshape(-1)]
    print(f"
RIPPER Test Accuracy: {accuracy_score(y_test_rip, rip_preds)*100:.2f}%")
except ImportError:
    print("wittgenstein not installed (pip install wittgenstein), proceeding directly with FOIL implementation below.")

In [ ]:
# FOIL Algorithm implementation from scratch using FOIL Information Gain
def foil_gain(p0, n0, p1, n1):
    if p1 == 0:
        return -float("inf")
    # gain = p1 * (log2(p1 / (p1 + n1)) - log2(p0 / (p0 + n0)))
    gain = p1 * (math.log2((p1 / (p1 + n1)) + 1e-9) - math.log2((p0 / (p0 + n0)) + 1e-9))
    return gain

def learn_foil(data, target_col, pos_label, max_rules=4, max_literals=3):
    remaining = data.copy()
    rules = []
    
    for r_idx in range(max_rules):
        pos_rem = (remaining[target_col] == pos_label).sum()
        if pos_rem == 0:
            break
            
        current_data = remaining.copy()
        rule_literals = []
        
        for _ in range(max_literals):
            p0 = (current_data[target_col] == pos_label).sum()
            n0 = (current_data[target_col] != pos_label).sum()
            if n0 == 0:
                break # pure rule
                
            best_gain = -float("inf")
            best_lit = None
            best_mask = None
            
            for f in feat_cols:
                for val in current_data[f].unique():
                    mask = current_data[f] == val
                    sub = current_data[mask]
                    p1 = (sub[target_col] == pos_label).sum()
                    n1 = (sub[target_col] != pos_label).sum()
                    gain = foil_gain(p0, n0, p1, n1)
                    if gain > best_gain:
                        best_gain = gain
                        best_lit = (f, val)
                        best_mask = mask
                        
            if best_lit is None or best_gain <= 0:
                break
                
            rule_literals.append(best_lit)
            current_data = current_data[best_mask]
            
        if not rule_literals:
            break
            
        # remove positives covered by this rule
        rule_mask = pd.Series([True] * len(remaining), index=remaining.index)
        for f, v in rule_literals:
            rule_mask &= (remaining[f] == v)
        remaining = remaining[~(rule_mask & (remaining[target_col] == pos_label))]
        rules.append(rule_literals)
        
    return rules

# train FOIL
foil_rules = learn_foil(train_df, "target", "malignant")

print("Learned FOIL Rules for Target=Malignant:")
for idx, r in enumerate(foil_rules, 1):
    cond = " AND ".join(f"{f}={v}" for f, v in r)
    print(f"  Rule {idx}: IF {cond} THEN target=malignant")

In [ ]:
# evaluate FOIL rules on test data
def predict_foil(rules, X_test, default_label="benign"):
    predictions = []
    for _, row in X_test.iterrows():
        matched = False
        for r in rules:
            if all(row[f] == v for f, v in r):
                predictions.append("malignant")
                matched = True
                break
        if not matched:
            predictions.append(default_label)
    return predictions

test_preds = predict_foil(foil_rules, test_df)
test_acc = accuracy_score(test_df["target"], test_preds)

print(f"FOIL Test Accuracy: {test_acc*100:.2f}%
")
print("Classification Report:")
print(classification_report(test_df["target"], test_preds, zero_division=0))